In [1]:
!pip list


Package                   Version
------------------------- ----------
absl-py                   2.5.1
anyio                     4.15.1
argon2-cffi               25.1.0
argon2-cffi-bindings      26.1.0
arrow                     1.4.0
asttokens                 3.0.2
async-lru                 2.4.0
attrs                     26.1.0
babel                     2.18.0
beautifulsoup4            4.15.0
bleach                    6.4.0
certifi                   2026.10.23
cffi                      2.1.1
charset-normalizer        3.5.2
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.4.0
cycler                    0.12.1
debugpy                   1.8.22
defusedxml                0.7.1
executing                 2.3.0
fastjsonschema            2.22.2
flatbuffers               25.12.19
fonttools                 4.68.0
fqdn                      1.6.0
h11                       0.16.0
httpcore                  1.0.9
httpx                     0.28.1
idna         


[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def normalize_landmarks(landmarks_63):
    pts = np.array(landmarks_63).reshape(21, 3)
    pts = pts - pts[0]
    max_val = np.max(np.abs(pts))
    if max_val > 0: pts = pts / max_val
    return pts.flatten()

def augment_landmarks(pts_flat, angle_deg=15, noise_std=0.015):
    pts = pts_flat.reshape(21, 3).copy()
    theta = np.radians(np.random.uniform(-angle_deg, angle_deg))
    c, s = np.cos(theta), np.sin(theta)
    rot_matrix = np.array([[c, -s], [s, c]])
    pts[:, :2] = pts[:, :2] @ rot_matrix.T
    pts += np.random.normal(0, noise_std, pts.shape)
    return pts.flatten()

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

# YOUR TWO DATASET PATHS
dataset_paths = [
    r"D:\Signverse\DATA",              
    r"D:\Signverse\MY_WEBCAM_DATA"     
]

data = []
labels = []
image_paths = []

print("Processing and Augmenting all data...")

for dataset_path in dataset_paths:
    if not os.path.exists(dataset_path): 
        print(f"Warning: Folder not found -> {dataset_path}")
        continue
    
    for folder_name in os.listdir(dataset_path):
        folder_path = os.path.join(dataset_path, folder_name)
        if os.path.isdir(folder_path):
            for image_name in os.listdir(folder_path):
                image_path = os.path.join(folder_path, image_name)
                img = cv2.imread(image_path)
                if img is None: continue
                    
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                results = hands.process(img_rgb)
                
                if results.multi_hand_landmarks:
                    hand_landmarks = results.multi_hand_landmarks[0]
                    raw_landmarks = []
                    for lm in hand_landmarks.landmark:
                        raw_landmarks.extend([lm.x, lm.y, lm.z])
                    
                    normalized = normalize_landmarks(raw_landmarks)
                    
                    # Save Original
                    data.append(normalized)
                    labels.append(folder_name)
                    image_paths.append(image_path)
                    
                    # Save 8 Augmented Versions
                    for _ in range(8):
                        data.append(augment_landmarks(normalized))
                        labels.append(folder_name)
                        image_paths.append(image_path + "_aug")

hands.close()

df = pd.DataFrame(data)
df['Label'] = labels
df['Image_Path'] = image_paths
df.to_csv('master_custom_data.csv', index=False)

print("=====================================")
print(f"Total rows created: {len(data)}")
print("Saved to master_custom_data.csv")
print("=====================================")

Processing and Augmenting all data...
Total rows created: 2241
Saved to master_custom_data.csv


In [1]:
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 1. Normalization Function
def normalize_landmarks(landmarks_63):
    pts = np.array(landmarks_63).reshape(21, 3)
    pts = pts - pts[0] # Center relative to wrist
    max_val = np.max(np.abs(pts))
    if max_val > 0: 
        pts = pts / max_val # Scale invariant
    return pts.flatten()

# 2. Augmentation Function
def augment_landmarks(pts_flat, angle_deg=15, noise_std=0.015):
    pts = pts_flat.reshape(21, 3).copy()
    # Random 2D rotation
    theta = np.radians(np.random.uniform(-angle_deg, angle_deg))
    c, s = np.cos(theta), np.sin(theta)
    rot_matrix = np.array([[c, -s], [s, c]])
    pts[:, :2] = pts[:, :2] @ rot_matrix.T
    # Add slight noise
    pts += np.random.normal(0, noise_std, pts.shape)
    return pts.flatten()

# 3. Setup MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

# 4. Define your dataset paths
dataset_paths = [
    r"D:\Signverse\DATA",               # Your friends' images
    r"D:\Signverse\MY_WEBCAM_DATA"      # Your webcam images (A-D + new letters)
]

data = []
labels = []
image_paths = []

print("Processing and Augmenting all data...")

# 5. Main Loop
for dataset_path in dataset_paths:
    if not os.path.exists(dataset_path): 
        print(f"Warning: Folder not found -> {dataset_path}")
        continue
    
    for folder_name in os.listdir(dataset_path):
        folder_path = os.path.join(dataset_path, folder_name)
        if os.path.isdir(folder_path):
            for image_name in os.listdir(folder_path):
                image_path = os.path.join(folder_path, image_name)
                
                # Read image
                img = cv2.imread(image_path)
                if img is None: continue
                    
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                results = hands.process(img_rgb)
                
                # If hand is detected
                if results.multi_hand_landmarks:
                    hand_landmarks = results.multi_hand_landmarks[0]
                    raw_landmarks = []
                    for lm in hand_landmarks.landmark:
                        raw_landmarks.extend([lm.x, lm.y, lm.z])
                    
                    # A. Normalize the landmarks
                    normalized = normalize_landmarks(raw_landmarks)
                    
                    # B. Save the original image data
                    data.append(normalized)
                    labels.append(folder_name)
                    image_paths.append(image_path)
                    
                    # C. Save 8 Augmented (rotated/noisy) versions
                    for _ in range(8):
                        data.append(augment_landmarks(normalized))
                        labels.append(folder_name)
                        image_paths.append(image_path + "_aug")

hands.close()

# 6. Save to CSV
df = pd.DataFrame(data)
df['Label'] = labels
df['Image_Path'] = image_paths
df.to_csv('master_custom_data.csv', index=False)

print("=====================================")
print(f"SUCCESS: Total rows created = {len(data)}")
print("Saved to master_custom_data.csv")
print("=====================================")

Processing and Augmenting all data...
SUCCESS: Total rows created = 2781
Saved to master_custom_data.csv


In [1]:

import sys
import importlib.metadata as metadata

print("Python executable:", sys.executable)
print("Python version:", sys.version)

packages = [
    "mediapipe",
    "opencv-contrib-python",
    "opencv-python",
    "numpy",
    "scikit-learn",
    "tensorflow",
    "protobuf",
    "jupyter"
]

print("\nINSTALLED PACKAGES")

for package in packages:
    try:
        print(package, ":", metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, ": NOT INSTALLED")

print("\nMEDIAPIPE TEST")

try:
    import mediapipe as mp
    print("MediaPipe imported successfully")
    print("Solutions available:", hasattr(mp, "solutions"))
except Exception as e:
    print("MediaPipe error:", repr(e))


Python executable: D:\Datastets\python.exe
Python version: 3.12.0rc3 (tags/v3.12.0rc3:b973ab3, Sep 19 2023, 09:39:54) [MSC v.1935 64 bit (AMD64)]

INSTALLED PACKAGES
mediapipe : 0.10.14
opencv-contrib-python : 5.0.0.93
opencv-python : 5.0.0.93
numpy : 2.5.4
scikit-learn : 1.9.1
tensorflow : NOT INSTALLED
protobuf : 4.25.9
jupyter : 1.1.1

MEDIAPIPE TEST
MediaPipe imported successfully
Solutions available: True
